# ResNet-50 Classification Baseline

AIML339, Anjola Sigbeku (300665808), Victoria University of Wellington.

The design report proposed a ResNet-50 baseline. The project became a
detection study, so a like-for-like comparison is impossible: a classifier
cannot find pieces, and mAP is not an accuracy. What is comparable, and more
interesting, is **how much each approach loses across the synthetic-to-real
domain shift**.

The classifier is handed ground-truth boxes, so localisation is free. Its
drop from synthetic to real therefore isolates the classification half of the
problem, which is exactly what finding F7 claimed the domain shift damages.

**Before running:** *Runtime -> Change runtime type -> T4 GPU*.

Total runtime is roughly 1.5 hours: about 40 minutes of downloading, 10
minutes cutting crops, and 30 minutes training.

One confound is handled explicitly rather than disclaimed. chesscog boxes are
**amodal**, computed from all mesh vertices including occluded ones, so a
synthetic crop contains whatever stands in front of the piece. ChessReD boxes
are tight on the visible piece. Section 8 measures that difference instead of
arguing about it.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

import torch
print()
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise SystemExit(
        'No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.'
    )

## 2. Mount Drive

Checkpoints and reports go here so a reclaimed session costs one epoch
rather than the whole run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/AIML339/chess-detection')
RUNS = DRIVE / 'runs' / 'resnet'
REPORTS = DRIVE / 'reports'
for d in (RUNS, REPORTS):
    d.mkdir(parents=True, exist_ok=True)
print('runs    ->', RUNS)
print('reports ->', REPORTS)

## 3. Clone the repository (private)

Same token setup as `train_colab.ipynb`. If `GITHUB_TOKEN` is already in
Colab Secrets from that notebook, this cell just works.

In [ ]:
import subprocess
from pathlib import Path

OWNER = 'Logic911e'
REPO_NAME = 'chess-piece-detection'
DEST = Path('/content') / REPO_NAME

try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None

public_url = f'https://github.com/{OWNER}/{REPO_NAME}.git'
clone_url = (
    f'https://{TOKEN}@github.com/{OWNER}/{REPO_NAME}.git' if TOKEN
    else public_url
)

if DEST.exists():
    print('already cloned, pulling')
    subprocess.run(['git', '-C', str(DEST), 'pull', '--ff-only'], check=False)
else:
    r = subprocess.run(['git', 'clone', '--depth', '1', clone_url, str(DEST)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        # Never print the token back out in an error message.
        msg = r.stderr.replace(TOKEN, '***') if TOKEN else r.stderr
        raise SystemExit(f'clone failed:\n{msg}')
    # Drop the credential from the stored remote so later git commands and
    # any printed output cannot leak it.
    subprocess.run(['git', '-C', str(DEST), 'remote', 'set-url',
                    'origin', public_url], check=True)
    print('cloned')

%cd /content/chess-piece-detection
!git log --oneline -1

## 4. Install dependencies

In [ ]:
!pip install -q osfclient

import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__)

## 5. Download both datasets

Synthetic renders from OSF (~6.9 GB of archives, 4,888 images) and
ChessReD2K from 4TU (4.68 GB, 2,078 box-annotated photographs). Both
downloaders resume, so re-running this cell after a drop is safe.

In [ ]:
!python src/download_synth.py --clean

In [ ]:
!python src/download_chessred.py --images --extract

## 6. Convert both to YOLO format and audit

The audits exit non-zero on any hard error, so a bad conversion stops here
rather than surfacing later as an inexplicably poor model. The king check is
the load-bearing one: every legal position has exactly one king per side, so
`white-king` and `black-king` must each total the image count.

In [ ]:
!python src/synth_to_yolo.py --src data/raw/render --dest data/processed/synth
!python src/inspect_data.py --data data/processed/synth --json reports/audit.json

In [ ]:
!python src/chessred_to_yolo.py --src data/raw/chessred --dest data/processed/chessred2k
!python src/inspect_data.py --data data/processed/chessred2k --json reports/audit_chessred.json

## 7. Cut per-piece crops

Roughly 105,000 synthetic crops and 30,000 real ones, each letterboxed to
128px square. Letterboxing rather than squashing is deliberate: a rook is
squat and a king is tall, and anisotropic resizing would throw that away.

In [ ]:
!python src/make_crops.py --src data/processed/synth --dest data/crops/synth
!python src/make_crops.py --src data/processed/chessred2k --dest data/crops/chessred2k

## 8. Measure the amodal-box confound first

Before the headline number, quantify how much of any gap is box convention
rather than domain shift. `--shrink 0.8` keeps the middle 80% of each box,
approximating a modal box on the amodal synthetic side.

If the synthetic-to-real gap is much smaller with shrunk boxes, then part of
what looks like a domain gap is really an annotation-convention gap, and that
has to be said plainly in the report.

In [ ]:
!python src/make_crops.py --src data/processed/synth \
    --dest data/crops/synth_modal --shrink 0.8

## 9. Train on synthetic crops

Selection is on the synthetic **val** split. The two test sets passed with
`--eval` are never consulted during training, for the same reason the
detector's fine-tuning validates on val: choosing a checkpoint on test would
leak it into every number reported from it.

In [ ]:
!python src/resnet_baseline.py \
    --train data/crops/synth/train \
    --val data/crops/synth/val \
    --eval synthetic=data/crops/synth/test \
    --eval real=data/crops/chessred2k/test \
    --epochs 10 --batch 128 --workers 2 \
    --tag synth --seed 42 \
    --out '/content/drive/MyDrive/AIML339/chess-detection/runs/resnet'

## 10. The same model trained on shrunk (modal) synthetic boxes

The comparison that separates domain shift from box convention.

In [ ]:
!python src/resnet_baseline.py \
    --train data/crops/synth_modal/train \
    --val data/crops/synth_modal/val \
    --eval synthetic=data/crops/synth_modal/test \
    --eval real=data/crops/chessred2k/test \
    --epochs 10 --batch 128 --workers 2 \
    --tag synth-modal --seed 42 \
    --out '/content/drive/MyDrive/AIML339/chess-detection/runs/resnet'

## 11. Data-efficiency curve, indexed by photographs

`--limit-images N` samples N source **photographs**, not N crops, stratified
across games. This matters: the detector's curve is indexed by photographs,
and indexing the classifier by crops would hand it roughly thirty times the
data at every point and make the comparison meaningless.

Three seeds at each N, matching the detector sweep, because F12 showed the
spread at small N is large enough to invert conclusions.

In [ ]:
import subprocess

SYNTH_CKPT = ('/content/drive/MyDrive/AIML339/chess-detection/'
              'runs/resnet/synth-seed42/best.pt')
OUT = '/content/drive/MyDrive/AIML339/chess-detection/runs/resnet'

for n in (2, 5, 10, 25, 50):
    for seed in (42, 43, 44):
        print(f'\n{"=" * 60}\nN={n}  seed={seed}\n{"=" * 60}', flush=True)
        subprocess.run([
            'python', 'src/resnet_baseline.py',
            '--init', SYNTH_CKPT,
            '--train', 'data/crops/chessred2k/train',
            '--val', 'data/crops/chessred2k/val',
            '--eval', 'real=data/crops/chessred2k/test',
            '--limit-images', str(n),
            '--epochs', '15', '--batch', '64', '--workers', '2',
            '--tag', f'real-n{n}', '--seed', str(seed),
            '--out', OUT,
        ], check=False)

## 12. Collect the results

Gathers every `report.json` into one table for the evaluation report.

In [ ]:
import json, csv
from pathlib import Path

OUT = Path('/content/drive/MyDrive/AIML339/chess-detection/runs/resnet')
rows = []
for rep in sorted(OUT.glob('*/report.json')):
    d = json.loads(rep.read_text())
    for name, m in d['results'].items():
        rows.append({
            'run': d['tag'], 'seed': d['seed'], 'eval_set': name,
            'train_images': d.get('train_images'),
            'train_crops': d['train_crops'],
            'top1': round(m['top1'], 4),
            'balanced_acc': round(m['balanced_acc'], 4),
            'macro_f1': round(m['macro_f1'], 4),
            'type_acc': round(m['type_acc'], 4),
            'colour_acc': round(m['colour_acc'], 4),
        })

dest = Path('/content/drive/MyDrive/AIML339/chess-detection/reports/resnet_baseline.csv')
if rows:
    with dest.open('w', newline='') as fh:
        wr = csv.DictWriter(fh, fieldnames=list(rows[0]))
        wr.writeheader()
        wr.writerows(rows)

hdr = f"{'run':<14}{'seed':>5}{'set':>11}{'imgs':>6}{'top1':>8}{'bal':>8}{'type':>8}{'colour':>8}"
print(hdr)
print('-' * len(hdr))
for r in rows:
    print(f"{r['run']:<14}{r['seed']:>5}{r['eval_set']:>11}"
          f"{str(r['train_images'] or '-'):>6}{r['top1']:>8.4f}"
          f"{r['balanced_acc']:>8.4f}{r['type_acc']:>8.4f}{r['colour_acc']:>8.4f}")
print(f'\nwrote {dest}')